Read in all files 

In [69]:
import pandas as pd
from pathlib import Path
import numpy as np


#read input files 
IO_file = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\MRIO\Figaro\Industry\flatfile_eu-ic-io_ind-by-ind_25ed_2023\flatfile_eu-ic-io_ind-by-ind_25ed_2023.csv") #input output tables for industry
IO_matrix = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\MRIO\Figaro\Industry\matrix_eu-ic-io_ind-by-ind_25ed_2023.csv")
GHG_file = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\MRIO\Figaro\Emissions\env_ac_ghgfp_2023_cleaned.csv") #Emissions 
IMPORT_file = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\MRIO\Figaro\Product\EU_RES_IMPORT_FILE.csv") #EU imported renewable energy sources
region_file = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\MRIO\Figaro\regions.csv") #csv for mapping regions to correct names
buildout_linear = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\Material\Buildout_cost_opt\buildout_linear.csv") #Linear buildout till 2050
buildout_gompertz = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\Material\Buildout_cost_opt\buildout_gompertz.csv") #Gompertz growth buildout till 2050
buildout_logistics = Path(r"C:\Users\menne\Documents\Master\Thesis\Data\Material\Buildout_cost_opt\buildout_logistics.csv") #Logistics growth buildout till 2050

df_IO = pd.read_csv(IO_file, sep=",")
df_matrix = pd.read_csv(IO_matrix)
df_GHG = pd.read_csv(GHG_file, sep=";", encoding="utf-8-sig", low_memory=False)
df_IMPORT = pd.read_csv(IMPORT_file, sep=",", encoding="utf-8-sig", low_memory=False)
df_regions = pd.read_csv(region_file, sep=";", encoding="utf-8-sig", low_memory=False)
df_linear_buildout = pd.read_csv(buildout_linear)
df_gompertz_buildout = pd.read_csv(buildout_gompertz)
df_logistics_buildout = pd.read_csv(buildout_logistics)

In [70]:
# Give countries the right names & only take into account the 46 individual countries (code other countries as Rest Of World)
# Input/Output file
io_to_iso3 = (
    df_regions
    .dropna(subset=["IO", "ISO_3"])
    .set_index("IO")["ISO_3"]
    .to_dict()
)

for column in ["refArea", "counterpartArea"]:
    df_IO[column] = df_IO[column].map(io_to_iso3).fillna("ROW")
    
    
# Matrix version

df_matrix = df_matrix.set_index("rowLabels")
df_matrix.index = df_matrix.index.astype(str).str.strip()
df_matrix.columns = df_matrix.columns.astype(str).str.strip()
df_matrix = df_matrix.apply(
    pd.to_numeric,
    errors="coerce"
).fillna(0)

def map_io_label(label, mapping):
    label = str(label).strip()
    country, sector = str(label).split("_", 1)
    country_iso3 = mapping.get(country, "ROW")
    return f"{country_iso3}_{sector}"


df_matrix.index = [
    map_io_label(label, io_to_iso3)
    for label in df_matrix.index
]

df_matrix.columns = [
    map_io_label(label, io_to_iso3)
    for label in df_matrix.columns
]
    
    
# GHG file    
GHG_to_iso3 = (
    df_regions
    .dropna(subset=["GHG", "ISO_3"])
    .set_index("GHG")["ISO_3"]
    .to_dict()
)

for column in ["c_dest", "c_orig"]:
    df_GHG[column] = df_GHG[column].map(GHG_to_iso3).fillna("ROW")
    
# Import file
IMPORT_to_iso3 = (
    df_regions
    .dropna(subset=["Import", "ISO_3"])
    .set_index("Import")["ISO_3"]
    .to_dict()
)

for column in ["reporter", "partner"]:
    df_IMPORT[column] = df_IMPORT[column].map(IMPORT_to_iso3).fillna("ROW")
    
# Remove aggregates
io_aggregates_refArea = [
    "W2"
]

ghg_aggregates_c_dest = [
    "EXTRA-EU-27",
    "EU-27"
]

ghg_aggregates_c_orig = [
    "EXTRA-EU-27",
    "EU-27",
    "ALL"
]

import_aggregates_partner = [
    "EXTRA-EU-27",
    "EXTRA-EA-21",
    "EXTRA-EA",
    "INT-EA"
]

import_product = [
    "Wood pellets"
]

# Remove from IO file
df_IO = df_IO.loc[
    ~df_IO["refArea"].isin(io_aggregates_refArea)
].copy()


# Remove from GHG file
df_GHG = df_GHG.loc[
    (~df_GHG["c_dest"].isin(ghg_aggregates_c_dest)) &
    (~df_GHG["c_orig"].isin(ghg_aggregates_c_orig))
].copy()


# Remove from import file
df_IMPORT = df_IMPORT.loc[
    (~df_IMPORT["partner"].isin(import_aggregates_partner))&
    (~df_IMPORT["product"].isin(import_product))
].copy()
    
    
# Group products into tech sources    
products_to_techs = {
    #"Wood pellets" : "bio fuel",
    "Photovoltaic cells assembled in modules or made up into panels" : "solar",
    "Photovoltaic cells not assembled in modules or made up into panels" : "solar",
    "Static converters" : "solar",
    "Electric accumulators (excl. spent and lead-acid, nickel-cadmium, nickel-iron, nickel-metal hydride and lithium-ion accumulators)(2022-2500);Electric accumulators (excl. spent and lead-acid, nickel-cadmium, nickel-iron, nickel-metal hydride and lithium-ion accumulators)(2012-2021)" : "batteries",
    "Lithium-ion accumulators (excl. spent)" : "batteries",
    "Generating sets, wind-powered(2002-2500);Generating sets, wind-powered(1996-2001)" : "wind"
}

df_IMPORT["product"] = df_IMPORT["product"].replace(products_to_techs)

In [71]:
# Add up all rows with same values
# Input/Output file
df_IO["obsValue"] = pd.to_numeric(
    df_IO["obsValue"],
    errors="coerce"
).fillna(0)

df_IO = (
    df_IO
    .groupby(
        ["refArea", "rowIi", "counterpartArea", "colIi"],
        as_index=False,
        dropna=False
    )["obsValue"]
    .sum()
)



df_IO["icioiRow"] = (
    df_IO["refArea"].astype(str)
    + "_"
    + df_IO["rowIi"].astype(str)
)

df_IO["icioiCol"] = (
    df_IO["counterpartArea"].astype(str)
    + "_"
    + df_IO["colIi"].astype(str)
)

df_IO = df_IO[
    [
        "icioiRow",
        "icioiCol",
        "refArea",
        "rowIi",
        "counterpartArea",
        "colIi",
        "obsValue"
    ]
]


# Matrix file
df_matrix = df_matrix.groupby(df_matrix.index).sum()
df_matrix = df_matrix.T.groupby(df_matrix.columns).sum().T


# GHG file
df_GHG["OBS_VALUE"] = (
    df_GHG["OBS_VALUE"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

df_GHG["OBS_VALUE"] = pd.to_numeric(
    df_GHG["OBS_VALUE"],
    errors="coerce"
).fillna(0)

df_GHG = (
    df_GHG
    .groupby(
        ["c_dest", "na_item", "c_orig", "nace_r2", "nace_r2_code"],
        as_index=False,
        dropna=False
    )["OBS_VALUE"]
    .sum()
)

# Import file
df_IMPORT["OBS_VALUE"] = (
    df_IMPORT["OBS_VALUE"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

df_IMPORT["OBS_VALUE"] = pd.to_numeric(
    df_IMPORT["OBS_VALUE"],
    errors="coerce"
).fillna(0)

df_IMPORT = (
    df_IMPORT
    .groupby(
        ["reporter", "partner", "product"],
        as_index=False,
        dropna=False
    )["OBS_VALUE"]
    .sum()
)

EE-MRIO analysis

In [72]:
# Some model variables
EU_countries = [
    "AUT",  # Austria
    "BEL",  # Belgium
    "BGR",  # Bulgaria
    "CYP",  # Cyprus
    "CZE",  # Czechia
    "DEU",  # Germany
    "DNK",  # Denmark
    "ESP",  # Spain
    "EST",  # Estonia
    "FIN",  # Finland
    "FRA",  # France
    "GRC",  # Greece
    "HRV",  # Croatia
    "HUN",  # Hungary
    "IRL",  # Ireland
    "ITA",  # Italy
    "LVA",  # Latvia
    "LTU",  # Lithuania
    "LUX",  # Luxembourg
    "NLD",  # Netherlands
    "POL",  # Poland
    "PRT",  # Portugal
    "ROU",  # Romania
    "SVK",  # Slovakia
    "SVN",  # Slovenia
    "SWE",  # Sweden
] # Malta not included

dollar_to_euro_2023 = 0.9242

capacity_price = {"solar": 261e9 * dollar_to_euro_2023, "offshore wind": 700e9 * dollar_to_euro_2023, "onshore wind": 700e9 * dollar_to_euro_2023, "batteries": 260e9 * dollar_to_euro_2023} #Euro/(TW/TWh)
construction_price = {"solar": (758e9 - 261e9) * dollar_to_euro_2023, "offshore wind": (2800e9 - 700e9) * dollar_to_euro_2023, "onshore wind": (1160e9 - 700e9) * dollar_to_euro_2023, "batteries": (273e9 - 260e9) * dollar_to_euro_2023} #Euro/(TW/TWh)

final_demand = ["P3_S13", "P3_S14", "P3_S15", "P51G", "P5M"] #Nace codes for final demand

value_added = ["D1", "D21X31", "D29X39", "B2A3G", "OP_RES", "OP_NRES"] #Nace codes for value added

res_code = {"batteries" : "C27", "solar" : "C27", "wind": "C28", "construction" : "F"} #Nace codes for technologies (techs)

In [73]:
# Calculate leontief inverse values to determine upstream emissions
# Differentiate between Intermediate use bock & Final use block 
def get_sector(label):
    return str(label).split("_", 1)[1]


row_sectors = pd.Series(df_matrix.index, index=df_matrix.index).apply(get_sector)
col_sectors = pd.Series(df_matrix.columns, index=df_matrix.columns).apply(get_sector)

production_rows = row_sectors.loc[
    ~row_sectors.isin(value_added)
].index

sector_columns = col_sectors.loc[
    ~col_sectors.isin(final_demand)
].index

final_demand_columns = col_sectors.loc[
    col_sectors.isin(final_demand)
].index


# Build intermediate transaction matrix Z
Z = df_matrix.loc[
    production_rows,
    sector_columns
].copy()

# Make square matrix
all_sectors = sorted(set(Z.index) | set(Z.columns))

Z = Z.reindex(
    index=all_sectors,
    columns=all_sectors,
    fill_value=0
)

# Build final demand matrix Y
Y = df_matrix.loc[
    production_rows,
    final_demand_columns
].copy()

Y = Y.reindex(
    index=all_sectors,
    fill_value=0
)

# Calculate total output x
x_output = Z.sum(axis=1) + Y.sum(axis=1)

# Keep only sectors with positive output
valid_sectors = x_output[x_output > 0].index

Z = Z.reindex(
    index=valid_sectors,
    columns=valid_sectors,
    fill_value=0
)

Y = Y.reindex(
    index=valid_sectors,
    fill_value=0
)

x_output = x_output.reindex(valid_sectors)

# Calculate technical coefficient matrix A
A = Z.divide(x_output, axis=1).fillna(0)

# Calculate Leontief inverse
I = np.eye(len(A))

L = pd.DataFrame(
    np.linalg.solve(I - A.values, I),
    index=A.index,
    columns=A.columns
)

In [74]:
# Create flat dataframe for L: required_input, country_dest, sector_dest, country_orig, sector_orig
def split_sector_id(sector_id):
    country, sector = str(sector_id).split("_", 1)
    return country, sector

# Select only demand columns whose sector is in res_codes (used in analysis)
res_demand_columns = [
    col for col in L.columns
    if split_sector_id(col)[1] in set(res_code.values())
]

L_res = L.loc[:, res_demand_columns].copy()

df_L_res_flat = (
    L_res
    .stack()
    .reset_index()
)

df_L_res_flat.columns = [
    "required_sector_id",
    "demand_sector_id",
    "required_input"
]

df_L_res_flat[["country_dest", "sector_dest"]] = (
    df_L_res_flat["demand_sector_id"]
    .str.split("_", n=1, expand=True)
)
df_L_res_flat[["country_orig", "sector_orig"]] = (
    df_L_res_flat["required_sector_id"]
    .str.split("_", n=1, expand=True)
)

del df_L_res_flat["required_sector_id"]
del df_L_res_flat["demand_sector_id"]

In [75]:
# Dataframe with total emissions caused by each sector in each country: country_orig, sector, total_emissions
df_emissions = (
    df_GHG
    .loc[df_GHG["na_item"] == "Total"]
    .groupby(["c_orig", "nace_r2_code"], as_index=False)["OBS_VALUE"]
    .sum()
    .rename(columns={
            "c_orig": "country_orig",
            "nace_r2_code": "sector",
            "OBS_VALUE": "total_emissions"
        })
    
)

# Dataframe with final outputs by each sector in each country: country_orig, sector, total_output
df_total_output = (
    df_IO
    .groupby(["refArea", "rowIi"], as_index=False)["obsValue"]
    .sum()
    .rename(columns={
        "refArea": "country_orig",
        "rowIi": "sector",
        "obsValue": "total_output"
    })
)

# Dataframe with  direct emissions per euro for each sector in each country: country_orig, sector, total_emissions, total_output, direct_emissions_per_euro
df_direct_emissions_per_euro = df_emissions.merge(
    df_total_output[
        ["country_orig", "sector", "total_output"]
    ],
    on=["country_orig", "sector"],
    how="left"
)

df_direct_emissions_per_euro = df_direct_emissions_per_euro.rename(columns={
    "country_orig": "country"
})


df_direct_emissions_per_euro = df_direct_emissions_per_euro.dropna(
    subset=["total_output"]
).copy()

df_direct_emissions_per_euro = df_direct_emissions_per_euro.loc[
    df_direct_emissions_per_euro["total_output"] > 0
].copy()

df_direct_emissions_per_euro["direct_emissions_per_euro"] = df_direct_emissions_per_euro["total_emissions"] / df_direct_emissions_per_euro["total_output"]

In [76]:
# Dataframe with required emissions for 1 euro output from country_dest,sector_dest (upstream emissions)
# DF: required_input, country_dest, sector_dest, country_orig, sector_orig, direct_emissions_per_euro, upstream_emission_per_euro (direct_emissions_per_euro is for specific combination country_orig,sector_orig)
df_upstream_emissions_per_euro = df_L_res_flat.merge(
    df_direct_emissions_per_euro[
        ["country", "sector", "direct_emissions_per_euro"]
    ],
    left_on=["country_orig", "sector_orig"],
    right_on=["country", "sector"],
    how="left"
)


df_upstream_emissions_per_euro["upstream_emissions_per_euro"] = (
    df_upstream_emissions_per_euro["required_input"] *
    df_upstream_emissions_per_euro["direct_emissions_per_euro"]
)


# Dataframe with 
# Dataframe with total upstream emissions per euro for each country for each sector in the analysis
df_total_upstream_emissions_per_euro = (
    df_upstream_emissions_per_euro
    .groupby(["country_dest", "sector_dest"], as_index=False)["upstream_emissions_per_euro"]
    .sum()
    .rename(columns={
        "upstream_emissions_per_euro": "total_upstream_emissions_per_euro",
        "country_dest": "country",
        "sector_dest": "sector"
    }))

In [77]:
df_sector_total = (
    df_upstream_emissions_per_euro
    .groupby(["country_dest", "sector_dest", "sector_orig"], as_index=False)["upstream_emissions_per_euro"]
    .sum()
)

df_sector_shares = df_sector_total.merge(
    df_total_upstream_emissions_per_euro[
        ["country", "sector", "total_upstream_emissions_per_euro"]
    ],
    left_on=["country_dest", "sector_dest"],
    right_on=["country", "sector"],
    how="left"
)
df_sector_shares["sector_share"] = df_sector_shares["upstream_emissions_per_euro"]/df_sector_shares["total_upstream_emissions_per_euro"]

df_sector_shares = df_sector_shares[
    [
        "country_dest",
        "sector_dest",
        "sector_orig",
        "sector_share"
    ]
].copy()

In [78]:
df_upstream_emissions_per_euro_self = df_upstream_emissions_per_euro.loc[
    (df_upstream_emissions_per_euro["country_dest"] == df_upstream_emissions_per_euro["country_orig"])&
    (df_upstream_emissions_per_euro["sector_dest"].isin(res_code.values()))
].copy()


df_total_upstream_emissions_per_euro_self = (
    df_upstream_emissions_per_euro_self
    .groupby(["country_dest", "sector_dest"], as_index=False)["upstream_emissions_per_euro"]
    .sum()
    .rename(columns={
        "upstream_emissions_per_euro": "total_upstream_emissions_per_euro_self",
        "country_dest": "country",
        "sector_dest": "sector"
    }))



df_upstream_emissions_per_euro_EU = df_upstream_emissions_per_euro.loc[
    df_upstream_emissions_per_euro["country_orig"].isin(EU_countries)&
    (df_upstream_emissions_per_euro["sector_dest"].isin(res_code.values()))
].copy()

df_upstream_emissions_per_euro_EU = df_upstream_emissions_per_euro_EU[df_upstream_emissions_per_euro_EU["country_dest"] != df_upstream_emissions_per_euro_EU["country_orig"]]

df_total_upstream_emissions_per_euro_EU = (
    df_upstream_emissions_per_euro_EU
    .groupby(["country_dest", "sector_dest"], as_index=False)["upstream_emissions_per_euro"]
    .sum()
    .rename(columns={
        "upstream_emissions_per_euro": "total_upstream_emissions_per_euro_EU",
        "country_dest": "country",
        "sector_dest": "sector"
    }))

df_upstream_emissions_per_euro_non_EU = df_upstream_emissions_per_euro.loc[
    ~df_upstream_emissions_per_euro["country_orig"].isin(EU_countries)&
    (df_upstream_emissions_per_euro["sector_dest"].isin(res_code.values()))
].copy()

df_upstream_emissions_per_euro_non_EU = df_upstream_emissions_per_euro_non_EU[df_upstream_emissions_per_euro_non_EU["country_dest"] != df_upstream_emissions_per_euro_non_EU["country_orig"]]

df_total_upstream_emissions_per_euro_non_EU = (
    df_upstream_emissions_per_euro_non_EU
    .groupby(["country_dest", "sector_dest"], as_index=False)["upstream_emissions_per_euro"]
    .sum()
    .rename(columns={
        "upstream_emissions_per_euro": "total_upstream_emissions_per_euro_non_EU",
        "country_dest": "country",
        "sector_dest": "sector"
    }))



In [79]:

df_upstream_emission_shares = (
    df_total_upstream_emissions_per_euro
    .merge(
        df_total_upstream_emissions_per_euro_self,
        on=["country", "sector"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        df_total_upstream_emissions_per_euro_EU,
        on=["country", "sector"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        df_total_upstream_emissions_per_euro_non_EU,
        on=["country", "sector"],
        how="left",
        validate="one_to_one"
    )
)

split_columns = [
    "total_upstream_emissions_per_euro_self",
    "total_upstream_emissions_per_euro_EU",
    "total_upstream_emissions_per_euro_non_EU"
]

df_upstream_emission_shares[split_columns] = (
    df_upstream_emission_shares[split_columns].fillna(0)
)

df_upstream_emission_shares["share_self"] = np.where(
    df_upstream_emission_shares["total_upstream_emissions_per_euro"] > 0,
    df_upstream_emission_shares["total_upstream_emissions_per_euro_self"] /
    df_upstream_emission_shares["total_upstream_emissions_per_euro"],
    0
)

df_upstream_emission_shares["share_EU"] = np.where(
    df_upstream_emission_shares["total_upstream_emissions_per_euro"] > 0,
    df_upstream_emission_shares["total_upstream_emissions_per_euro_EU"] /
    df_upstream_emission_shares["total_upstream_emissions_per_euro"],
    0
)

df_upstream_emission_shares["share_non_EU"] = np.where(
    df_upstream_emission_shares["total_upstream_emissions_per_euro"] > 0,
    df_upstream_emission_shares["total_upstream_emissions_per_euro_non_EU"] /
    df_upstream_emission_shares["total_upstream_emissions_per_euro"],
    0
)

In [80]:
# Dataframe with information on import share for each country for each tech: country_dest, country_orig, tech, import, total import, import_share
df_import_shares = (
    df_IMPORT
    .groupby(["reporter", "partner", "product"], as_index=False)["OBS_VALUE"]
    .sum()
    .rename(columns={
        "reporter": "country_dest",
        "partner": "country_orig",
        "product": "tech",
        "OBS_VALUE": "import"
    })
)

# Total import for each country for each tech
df_import_shares["total_import"] = (
df_import_shares
.groupby(["country_dest", "tech"])["import"]
.transform("sum")
)

#Share of origin country in total import
df_import_shares["import_share"] = np.where(
df_import_shares["total_import"] > 0,
df_import_shares["import"] / df_import_shares["total_import"],
np.nan
)

In [81]:
df_import_shares["sector"] = (
    df_import_shares["tech"].map(res_code)
)

df_upstream_emission_shares_no_F = df_upstream_emission_shares.loc[
    df_upstream_emission_shares["sector"] != "F"
].copy()

df_upstream_location_shares = df_import_shares.merge(
    df_upstream_emission_shares_no_F[
        [
            "country",
            "sector",
            "share_self",
            "share_EU",
            "share_non_EU"
        ]
    ],
    left_on=["country_orig", "sector"],
    right_on=["country", "sector"],
    how="left",
    validate="many_to_one"
)

df_upstream_location_shares["weighted_share_self"] = (
    df_upstream_location_shares["import_share"] *
    df_upstream_location_shares["share_self"]
)

df_upstream_location_shares["weighted_share_EU"] = (
    df_upstream_location_shares["import_share"] *
    df_upstream_location_shares["share_EU"]
)

df_upstream_location_shares["weighted_share_non_EU"] = (
    df_upstream_location_shares["import_share"] *
    df_upstream_location_shares["share_non_EU"]
)

df_upstream_location_shares = df_upstream_location_shares[
    [
        "country_dest",
        "country_orig",
        "tech",
        "sector",
        "import",
        "total_import",
        "import_share",
        "weighted_share_self",
        "weighted_share_EU",
        "weighted_share_non_EU"
    ]
].copy()

df_total_upstream_location_shares = (
    df_upstream_location_shares
    .groupby(["country_dest", "tech"], as_index=False)[
        ["weighted_share_self", "weighted_share_EU", "weighted_share_non_EU"]
    ]
    .sum()
)

df_total_upstream_location_shares_F = df_upstream_emission_shares.loc[
    df_upstream_emission_shares["sector"] == "F"
]

In [82]:
# Create dataframe with final upstream emissions per euro for each country & tech (based on import shares)
# Dataframe with weighted upstream emissions: country_dest, country_orig, tech, import, total import, import_share, sector, upstream_emissions_per_euro, weighted_upstream_emissions_per_euro (weighted_upstream_emissions_per_euro = upstream_emissions_per_euro * import_share)
df_weighted_upstream_emissions_per_euro = df_import_shares.copy()

df_weighted_upstream_emissions_per_euro["sector"] = (
    df_weighted_upstream_emissions_per_euro["tech"].map(res_code)
)

df_upstream_factor = df_total_upstream_emissions_per_euro.rename(columns={
    "country": "country_orig",
    "sector_dest": "sector",
    "total_upstream_emissions_per_euro": "upstream_emissions_per_euro"
})

df_weighted_upstream_emissions_per_euro = df_weighted_upstream_emissions_per_euro.merge(
    df_upstream_factor[
        ["country_orig", "sector", "upstream_emissions_per_euro"]
    ],
    on=["country_orig", "sector"],
    how="left",
    validate="many_to_one"
)

df_weighted_upstream_emissions_per_euro["weighted_upstream_emissions_per_euro"] = (
df_weighted_upstream_emissions_per_euro["import_share"] *
df_weighted_upstream_emissions_per_euro["upstream_emissions_per_euro"]
)

# Dataframe with final upstream emissions per euro for each country for each tech: country, tech, total_upstream_emissions_per_euro
df_final_upstream_emissions_per_euro = (
df_weighted_upstream_emissions_per_euro
.groupby(["country_dest", "tech"], as_index=False)["weighted_upstream_emissions_per_euro"]
.sum()
.rename(columns={
"country_dest": "country",
"weighted_upstream_emissions_per_euro": "total_upstream_emissions_per_euro"})
)

df_upstream_emissions_per_euro_F = df_upstream_emissions_per_euro.loc[
    df_upstream_emissions_per_euro["sector_dest"] == "F"
].copy()

df_final_upstream_emissions_per_euro_F = (
df_upstream_emissions_per_euro_F
.groupby(["country_dest", "sector_dest"], as_index=False)["upstream_emissions_per_euro"]
.sum()
.rename(columns={
"country_dest": "country",
"sector_dest": "sector",
"upstream_emissions_per_euro": "total_upstream_emissions_per_euro"})
)

In [83]:
# Dataframes with the total annual investment in euro for each EU country, for each tech (from 2026 to 2050)

# Annual investment with Gompertz growth model
df_annual_investment_gompertz = (
df_gompertz_buildout
.groupby(["country", "tech", "year"], as_index=False)["new_capacity_required_TW"]
.sum()
)

df_annual_investment_gompertz["capacity_price"] = (
df_annual_investment_gompertz["tech"].map(capacity_price)
)

df_annual_investment_gompertz["construction_price"] = (
df_annual_investment_gompertz["tech"].map(construction_price)
)


df_annual_investment_gompertz["investment"] = np.where(
df_annual_investment_gompertz["new_capacity_required_TW"] > 0,
df_annual_investment_gompertz["new_capacity_required_TW"] * df_annual_investment_gompertz["capacity_price"],
np.nan
)

df_annual_investment_gompertz["construction_cost"] = np.where(
df_annual_investment_gompertz["new_capacity_required_TW"] > 0,
df_annual_investment_gompertz["new_capacity_required_TW"] * df_annual_investment_gompertz["construction_price"],
np.nan
)


# Annual investment with Logistics growth model
df_annual_investment_logistics = (
df_logistics_buildout
.groupby(["country", "tech", "year"], as_index=False)["new_capacity_required_TW"]
.sum()
)

df_annual_investment_logistics["capacity_price"] = (
df_annual_investment_logistics["tech"].map(capacity_price)
)

df_annual_investment_logistics["construction_price"] = (
df_annual_investment_logistics["tech"].map(construction_price)
)

df_annual_investment_logistics["investment"] = np.where(
df_annual_investment_logistics["new_capacity_required_TW"] > 0,
df_annual_investment_logistics["new_capacity_required_TW"] * df_annual_investment_logistics["capacity_price"],
np.nan
)

df_annual_investment_logistics["construction_cost"] = np.where(
df_annual_investment_logistics["new_capacity_required_TW"] > 0,
df_annual_investment_logistics["new_capacity_required_TW"] * df_annual_investment_logistics["construction_price"],
np.nan
)



# Annual investment with linear growth model
df_annual_investment_linear = (
df_linear_buildout
.groupby(["country", "tech", "year"], as_index=False)["new_capacity_required_TW"]
.sum()
)

df_annual_investment_linear["capacity_price"] = (
df_annual_investment_linear["tech"].map(capacity_price)
)

df_annual_investment_linear["construction_price"] = (
df_annual_investment_linear["tech"].map(construction_price)
)

df_annual_investment_linear["investment"] = np.where(
df_annual_investment_linear["new_capacity_required_TW"] > 0,
df_annual_investment_linear["new_capacity_required_TW"] * df_annual_investment_linear["capacity_price"],
np.nan
)

df_annual_investment_linear["construction_cost"] = np.where(
df_annual_investment_linear["new_capacity_required_TW"] > 0,
df_annual_investment_linear["new_capacity_required_TW"] * df_annual_investment_linear["construction_price"],
np.nan
)

In [97]:
# Dataframe with annual imported emissions per country

# Replace "onshore wind" & "offshore wind" with wind (like in final_upstream_emissions_per_euro dataframe)
df_annual_investment_gompertz = df_annual_investment_gompertz.copy()
df_annual_investment_logistics = df_annual_investment_logistics.copy()
df_annual_investment_linear = df_annual_investment_linear.copy()

df_annual_investment_gompertz["tech"] = (
    df_annual_investment_gompertz["tech"]
    .replace({
        "onshore wind": "wind",
        "offshore wind": "wind"
    })
)
df_annual_investment_logistics["tech"] = (
    df_annual_investment_logistics["tech"]
    .replace({
        "onshore wind": "wind",
        "offshore wind": "wind"
    })
)
df_annual_investment_linear["tech"] = (
    df_annual_investment_linear["tech"]
    .replace({
        "onshore wind": "wind",
        "offshore wind": "wind"
    })
)


annual_imported_emissions_per_country_linear = (
    df_annual_investment_linear
    .merge(
        df_final_upstream_emissions_per_euro.rename(columns={
            "total_upstream_emissions_per_euro": "total_upstream_emissions_per_sector"
        }),
        on=["country", "tech"],
        how="left",
        validate="many_to_one"
    )
)

annual_imported_emissions_per_country_linear["investment"] = (
    annual_imported_emissions_per_country_linear["investment"].fillna(0)
)

annual_imported_emissions_per_country_linear["imported_emissions_tonnes"] = (
    (annual_imported_emissions_per_country_linear["investment"] *
    annual_imported_emissions_per_country_linear["total_upstream_emissions_per_sector"])
    /1000
)

annual_imported_emissions_per_country_linear = annual_imported_emissions_per_country_linear[
    [
        "country",
        "tech",
        "year",
        "investment",
        "total_upstream_emissions_per_sector",
        "imported_emissions_tonnes"
    ]
].copy()

annual_imported_emissions_per_country_gompertz = (
    df_annual_investment_gompertz
    .merge(
        df_final_upstream_emissions_per_euro.rename(columns={
            "total_upstream_emissions_per_euro": "total_upstream_emissions_per_sector"
        }),
        on=["country", "tech"],
        how="left",
        validate="many_to_one"
    )
)

annual_imported_emissions_per_country_gompertz["investment"] = (
    annual_imported_emissions_per_country_gompertz["investment"].fillna(0)
)

annual_imported_emissions_per_country_gompertz["imported_emissions_tonnes"] = (
    (annual_imported_emissions_per_country_gompertz["investment"] *
    annual_imported_emissions_per_country_gompertz["total_upstream_emissions_per_sector"])
    /1000
)

annual_imported_emissions_per_country_gompertz = annual_imported_emissions_per_country_gompertz[
    [
        "country",
        "tech",
        "year",
        "investment",
        "total_upstream_emissions_per_sector",
        "imported_emissions_tonnes"
    ]
].copy()

annual_imported_emissions_per_country_logistic = (
    df_annual_investment_logistics
    .merge(
        df_final_upstream_emissions_per_euro.rename(columns={
            "total_upstream_emissions_per_euro": "total_upstream_emissions_per_sector"
        }),
        on=["country", "tech"],
        how="left",
        validate="many_to_one"
    )
)

annual_imported_emissions_per_country_logistic["investment"] = (
    annual_imported_emissions_per_country_logistic["investment"].fillna(0)
)

annual_imported_emissions_per_country_logistic["imported_emissions_tonnes"] = (
    (annual_imported_emissions_per_country_logistic["investment"] *
    annual_imported_emissions_per_country_logistic["total_upstream_emissions_per_sector"])
    /1000
)

annual_imported_emissions_per_country_logistic = annual_imported_emissions_per_country_logistic[
    [
        "country",
        "tech",
        "year",
        "investment",
        "total_upstream_emissions_per_sector",
        "imported_emissions_tonnes"
    ]
].copy()



In [104]:
annual_imported_emissions_location_linear = (
    annual_imported_emissions_per_country_linear
    .merge(
        df_total_upstream_location_shares.rename(columns={
            "country_dest": "country"
        }),
        on=["country", "tech"],
        how="left",
        validate="many_to_one"
    )
)

# ------------------------------------------------------------
# Calculate imported emissions by location
# ------------------------------------------------------------

annual_imported_emissions_location_linear["imported_emissions_tonnes_self"] = (
    annual_imported_emissions_location_linear["imported_emissions_tonnes"] *
    annual_imported_emissions_location_linear["weighted_share_self"]
)

annual_imported_emissions_location_linear["imported_emissions_tonnes_EU"] = (
    annual_imported_emissions_location_linear["imported_emissions_tonnes"] *
    annual_imported_emissions_location_linear["weighted_share_EU"]
)

annual_imported_emissions_location_linear["imported_emissions_tonnes_non_EU"] = (
    annual_imported_emissions_location_linear["imported_emissions_tonnes"] *
    annual_imported_emissions_location_linear["weighted_share_non_EU"]
)

annual_imported_emissions_location_linear["imported_emissions_tonnes_total"] = (
    annual_imported_emissions_location_linear["imported_emissions_tonnes"]
)

# ------------------------------------------------------------
# Keep requested columns
# ------------------------------------------------------------

annual_imported_emissions_location_linear = annual_imported_emissions_location_linear[
    [
        "country",
        "tech",
        "year",
        "imported_emissions_tonnes_self",
        "imported_emissions_tonnes_EU",
        "imported_emissions_tonnes_non_EU",
        "imported_emissions_tonnes_total"
    ]
].copy()


annual_imported_emissions_location_logistic = (
    annual_imported_emissions_per_country_logistic
    .merge(
        df_total_upstream_location_shares.rename(columns={
            "country_dest": "country"
        }),
        on=["country", "tech"],
        how="left",
        validate="many_to_one"
    )
)

# ------------------------------------------------------------
# Calculate imported emissions by location
# ------------------------------------------------------------

annual_imported_emissions_location_logistic["imported_emissions_tonnes_self"] = (
    annual_imported_emissions_location_logistic["imported_emissions_tonnes"] *
    annual_imported_emissions_location_logistic["weighted_share_self"]
)

annual_imported_emissions_location_logistic["imported_emissions_tonnes_EU"] = (
    annual_imported_emissions_location_logistic["imported_emissions_tonnes"] *
    annual_imported_emissions_location_logistic["weighted_share_EU"]
)

annual_imported_emissions_location_logistic["imported_emissions_tonnes_non_EU"] = (
    annual_imported_emissions_location_logistic["imported_emissions_tonnes"] *
    annual_imported_emissions_location_logistic["weighted_share_non_EU"]
)

annual_imported_emissions_location_logistic["imported_emissions_tonnes_total"] = (
    annual_imported_emissions_location_logistic["imported_emissions_tonnes"]
)

# ------------------------------------------------------------
# Keep requested columns
# ------------------------------------------------------------

annual_imported_emissions_location_logistic = annual_imported_emissions_location_logistic[
    [
        "country",
        "tech",
        "year",
        "imported_emissions_tonnes_self",
        "imported_emissions_tonnes_EU",
        "imported_emissions_tonnes_non_EU",
        "imported_emissions_tonnes_total"
    ]
].copy()



annual_imported_emissions_location_gompertz = (
    annual_imported_emissions_per_country_gompertz
    .merge(
        df_total_upstream_location_shares.rename(columns={
            "country_dest": "country"
        }),
        on=["country", "tech"],
        how="left",
        validate="many_to_one"
    )
)

# ------------------------------------------------------------
# Calculate imported emissions by location
# ------------------------------------------------------------

annual_imported_emissions_location_gompertz["imported_emissions_tonnes_self"] = (
    annual_imported_emissions_location_gompertz["imported_emissions_tonnes"] *
    annual_imported_emissions_location_gompertz["weighted_share_self"]
)

annual_imported_emissions_location_gompertz["imported_emissions_tonnes_EU"] = (
    annual_imported_emissions_location_gompertz["imported_emissions_tonnes"] *
    annual_imported_emissions_location_gompertz["weighted_share_EU"]
)

annual_imported_emissions_location_gompertz["imported_emissions_tonnes_non_EU"] = (
    annual_imported_emissions_location_gompertz["imported_emissions_tonnes"] *
    annual_imported_emissions_location_gompertz["weighted_share_non_EU"]
)

annual_imported_emissions_location_gompertz["imported_emissions_tonnes_total"] = (
    annual_imported_emissions_location_gompertz["imported_emissions_tonnes"]
)

# ------------------------------------------------------------
# Keep requested columns
# ------------------------------------------------------------

annual_imported_emissions_location_gompertz = annual_imported_emissions_location_gompertz[
    [
        "country",
        "tech",
        "year",
        "imported_emissions_tonnes_self",
        "imported_emissions_tonnes_EU",
        "imported_emissions_tonnes_non_EU",
        "imported_emissions_tonnes_total"
    ]
].copy()

In [105]:
# Dataframe with import result totals

# Total imported emissions in the EU every year for every tech
df_total_imported_emissions_EU_linear = (
    annual_imported_emissions_location_linear
    .groupby(["tech", "year"], as_index=False)[["imported_emissions_tonnes_self","imported_emissions_tonnes_EU", "imported_emissions_tonnes_non_EU", "imported_emissions_tonnes_total"]]
    .sum()
    )

# Total imported emissions for each tech
df_total_imported_emissions_techs_Linear = (
    df_total_imported_emissions_EU_linear
    .groupby(["tech"], as_index=False)[["imported_emissions_tonnes_self","imported_emissions_tonnes_EU","imported_emissions_tonnes_non_EU", "imported_emissions_tonnes_total"]]
    .sum()
    .rename(columns={
        "imported_emissions_tonnes_total": "total_imported_emissions_tonnes (2026-2050)",
        "imported_emissions_tonnes_self" : "domesctic_emissions (2026-2050)",
        "imported_emissions_tonnes_EU": "EU_imported_emissions_tonnes (2026-2050)",
        "imported_emissions_tonnes_non_EU" : "non_EU_imported_emissions_tonnes (2026-2050)"
        
        })
    )

# Dataframe with import result totals

# Total imported emissions in the EU every year for every tech
df_total_imported_emissions_EU_logistic = (
    annual_imported_emissions_location_logistic
    .groupby(["tech", "year"], as_index=False)[["imported_emissions_tonnes_self","imported_emissions_tonnes_EU", "imported_emissions_tonnes_non_EU", "imported_emissions_tonnes_total"]]
    .sum()
    )

# Total imported emissions for each tech
df_total_imported_emissions_techs_logistic = (
    df_total_imported_emissions_EU_logistic
    .groupby(["tech"], as_index=False)[["imported_emissions_tonnes_self","imported_emissions_tonnes_EU","imported_emissions_tonnes_non_EU", "imported_emissions_tonnes_total"]]
    .sum()
    .rename(columns={
        "imported_emissions_tonnes_total": "total_imported_emissions_tonnes (2026-2050)",
        "imported_emissions_tonnes_self" : "domesctic_emissions (2026-2050)",
        "imported_emissions_tonnes_EU": "EU_imported_emissions_tonnes (2026-2050)",
        "imported_emissions_tonnes_non_EU" : "non_EU_imported_emissions_tonnes (2026-2050)"
        
        })
    )

# Total imported emissions in the EU every year for every tech
df_total_imported_emissions_EU_gompertz = (
    annual_imported_emissions_location_gompertz
    .groupby(["tech", "year"], as_index=False)[["imported_emissions_tonnes_self","imported_emissions_tonnes_EU", "imported_emissions_tonnes_non_EU", "imported_emissions_tonnes_total"]]
    .sum()
    )

# Total imported emissions for each tech
df_total_imported_emissions_techs_gompertz = (
    df_total_imported_emissions_EU_gompertz
    .groupby(["tech"], as_index=False)[["imported_emissions_tonnes_self","imported_emissions_tonnes_EU","imported_emissions_tonnes_non_EU", "imported_emissions_tonnes_total"]]
    .sum()
    .rename(columns={
        "imported_emissions_tonnes_total": "total_imported_emissions_tonnes (2026-2050)",
        "imported_emissions_tonnes_self" : "domesctic_emissions (2026-2050)",
        "imported_emissions_tonnes_EU": "EU_imported_emissions_tonnes (2026-2050)",
        "imported_emissions_tonnes_non_EU" : "non_EU_imported_emissions_tonnes (2026-2050)"
        
        })
    )

In [108]:
# Dataframe with annual construction emissions per country

# Replace "onshore wind" & "offshore wind" with wind (like in final_upstream_emissions_per_euro dataframe)
df_annual_investment_gompertz = df_annual_investment_gompertz.copy()
df_annual_investment_logistics = df_annual_investment_logistics.copy()
df_annual_investment_linear = df_annual_investment_linear.copy()

df_annual_investment_gompertz["tech"] = (
    df_annual_investment_gompertz["tech"]
    .replace({
        "onshore wind": "wind",
        "offshore wind": "wind"
    })
)
df_annual_investment_logistics["tech"] = (
    df_annual_investment_logistics["tech"]
    .replace({
        "onshore wind": "wind",
        "offshore wind": "wind"
    })
)
df_annual_investment_linear["tech"] = (
    df_annual_investment_linear["tech"]
    .replace({
        "onshore wind": "wind",
        "offshore wind": "wind"
    })
)


annual_construction_emissions_per_country_linear = (
    df_annual_investment_linear
    .merge(
        df_final_upstream_emissions_per_euro_F.rename(columns={
            "total_upstream_emissions_per_euro": "total_upstream_emissions_per_sector"
        }),
        on=["country"],
        how="left",
        validate="many_to_one"
    )
)

annual_construction_emissions_per_country_linear["construction_cost"] = (
    annual_construction_emissions_per_country_linear["construction_cost"].fillna(0)
)

annual_construction_emissions_per_country_linear["construction_emissions_tonnes"] = (
    (annual_construction_emissions_per_country_linear["construction_cost"] *
    annual_construction_emissions_per_country_linear["total_upstream_emissions_per_sector"])
    /1000
)

annual_construction_emissions_per_country_linear = annual_construction_emissions_per_country_linear[
    [
        "country",
        "tech",
        "year",
        "construction_cost",
        "total_upstream_emissions_per_sector",
        "construction_emissions_tonnes"
    ]
].copy()

annual_construction_emissions_per_country_logistic = (
    df_annual_investment_logistics
    .merge(
        df_final_upstream_emissions_per_euro_F.rename(columns={
            "total_upstream_emissions_per_euro": "total_upstream_emissions_per_sector"
        }),
        on=["country"],
        how="left",
        validate="many_to_one"
    )
)

annual_construction_emissions_per_country_logistic["construction_cost"] = (
    annual_construction_emissions_per_country_logistic["construction_cost"].fillna(0)
)

annual_construction_emissions_per_country_logistic["construction_emissions_tonnes"] = (
    (annual_construction_emissions_per_country_logistic["construction_cost"] *
    annual_construction_emissions_per_country_logistic["total_upstream_emissions_per_sector"])
    /1000
)

annual_construction_emissions_per_country_logistic = annual_construction_emissions_per_country_logistic[
    [
        "country",
        "tech",
        "year",
        "construction_cost",
        "total_upstream_emissions_per_sector",
        "construction_emissions_tonnes"
    ]
].copy()

annual_construction_emissions_per_country_gompertz = (
    df_annual_investment_gompertz
    .merge(
        df_final_upstream_emissions_per_euro_F.rename(columns={
            "total_upstream_emissions_per_euro": "total_upstream_emissions_per_sector"
        }),
        on=["country"],
        how="left",
        validate="many_to_one"
    )
)

annual_construction_emissions_per_country_gompertz["construction_cost"] = (
    annual_construction_emissions_per_country_gompertz["construction_cost"].fillna(0)
)

annual_construction_emissions_per_country_gompertz["construction_emissions_tonnes"] = (
    (annual_construction_emissions_per_country_gompertz["construction_cost"] *
    annual_construction_emissions_per_country_gompertz["total_upstream_emissions_per_sector"])
    /1000
)

annual_construction_emissions_per_country_gompertz = annual_construction_emissions_per_country_gompertz[
    [
        "country",
        "tech",
        "year",
        "construction_cost",
        "total_upstream_emissions_per_sector",
        "construction_emissions_tonnes"
    ]
].copy()

In [109]:
annual_construction_emissions_location_linear = (
    annual_construction_emissions_per_country_linear
    .merge(
        df_total_upstream_location_shares_F,
        on=["country"],
        how="left",
        validate="many_to_one"
    )
)

# ------------------------------------------------------------
# Calculate imported emissions by location
# ------------------------------------------------------------

annual_construction_emissions_location_linear["construction_emissions_tonnes_self"] = (
    annual_construction_emissions_location_linear["construction_emissions_tonnes"] *
    annual_construction_emissions_location_linear["share_self"]
)

annual_construction_emissions_location_linear["construction_emissions_tonnes_EU"] = (
    annual_construction_emissions_location_linear["construction_emissions_tonnes"] *
    annual_construction_emissions_location_linear["share_EU"]
)

annual_construction_emissions_location_linear["construction_emissions_tonnes_non_EU"] = (
    annual_construction_emissions_location_linear["construction_emissions_tonnes"] *
    annual_construction_emissions_location_linear["share_non_EU"]
)

annual_construction_emissions_location_linear["construction_emissions_tonnes_total"] = (
    annual_construction_emissions_location_linear["construction_emissions_tonnes"]
)

# ------------------------------------------------------------
# Keep requested columns
# ------------------------------------------------------------

annual_construction_emissions_location_linear = annual_construction_emissions_location_linear[
    [
        "country",
        "tech",
        "year",
        "construction_emissions_tonnes_self",
        "construction_emissions_tonnes_EU",
        "construction_emissions_tonnes_non_EU",
        "construction_emissions_tonnes_total"
    ]
].copy()

annual_construction_emissions_location_logistic = (
    annual_construction_emissions_per_country_logistic
    .merge(
        df_total_upstream_location_shares_F,
        on=["country"],
        how="left",
        validate="many_to_one"
    )
)

# ------------------------------------------------------------
# Calculate imported emissions by location
# ------------------------------------------------------------

annual_construction_emissions_location_logistic["construction_emissions_tonnes_self"] = (
    annual_construction_emissions_location_logistic["construction_emissions_tonnes"] *
    annual_construction_emissions_location_logistic["share_self"]
)

annual_construction_emissions_location_logistic["construction_emissions_tonnes_EU"] = (
    annual_construction_emissions_location_logistic["construction_emissions_tonnes"] *
    annual_construction_emissions_location_logistic["share_EU"]
)

annual_construction_emissions_location_logistic["construction_emissions_tonnes_non_EU"] = (
    annual_construction_emissions_location_logistic["construction_emissions_tonnes"] *
    annual_construction_emissions_location_logistic["share_non_EU"]
)

annual_construction_emissions_location_logistic["construction_emissions_tonnes_total"] = (
    annual_construction_emissions_location_logistic["construction_emissions_tonnes"]
)

# ------------------------------------------------------------
# Keep requested columns
# ------------------------------------------------------------

annual_construction_emissions_location_logistic = annual_construction_emissions_location_logistic[
    [
        "country",
        "tech",
        "year",
        "construction_emissions_tonnes_self",
        "construction_emissions_tonnes_EU",
        "construction_emissions_tonnes_non_EU",
        "construction_emissions_tonnes_total"
    ]
].copy()


annual_construction_emissions_location_gompertz = (
    annual_construction_emissions_per_country_gompertz
    .merge(
        df_total_upstream_location_shares_F,
        on=["country"],
        how="left",
        validate="many_to_one"
    )
)

# ------------------------------------------------------------
# Calculate imported emissions by location
# ------------------------------------------------------------

annual_construction_emissions_location_gompertz["construction_emissions_tonnes_self"] = (
    annual_construction_emissions_location_gompertz["construction_emissions_tonnes"] *
    annual_construction_emissions_location_gompertz["share_self"]
)

annual_construction_emissions_location_gompertz["construction_emissions_tonnes_EU"] = (
    annual_construction_emissions_location_gompertz["construction_emissions_tonnes"] *
    annual_construction_emissions_location_gompertz["share_EU"]
)

annual_construction_emissions_location_gompertz["construction_emissions_tonnes_non_EU"] = (
    annual_construction_emissions_location_gompertz["construction_emissions_tonnes"] *
    annual_construction_emissions_location_gompertz["share_non_EU"]
)

annual_construction_emissions_location_gompertz["construction_emissions_tonnes_total"] = (
    annual_construction_emissions_location_gompertz["construction_emissions_tonnes"]
)

# ------------------------------------------------------------
# Keep requested columns
# ------------------------------------------------------------

annual_construction_emissions_location_gompertz = annual_construction_emissions_location_gompertz[
    [
        "country",
        "tech",
        "year",
        "construction_emissions_tonnes_self",
        "construction_emissions_tonnes_EU",
        "construction_emissions_tonnes_non_EU",
        "construction_emissions_tonnes_total"
    ]
].copy()

In [110]:
# Dataframe with construction result totals

# Total imported emissions in the EU every year for every tech
df_total_construction_emissions_EU_linear = (
    annual_construction_emissions_location_linear
    .groupby(["tech", "year"], as_index=False)[["construction_emissions_tonnes_self","construction_emissions_tonnes_EU", "construction_emissions_tonnes_non_EU", "construction_emissions_tonnes_total"]]
    .sum()
    )

# Total imported emissions for each tech
df_total_construction_emissions_techs = (
    df_total_construction_emissions_EU_linear
    .groupby(["tech"], as_index=False)[["construction_emissions_tonnes_self","construction_emissions_tonnes_EU","construction_emissions_tonnes_non_EU", "construction_emissions_tonnes_total"]]
    .sum()
    .rename(columns={
        "construction_emissions_tonnes_total": "total_construction_emissions_tonnes (2026-2050)",
        "construction_emissions_tonnes_self" : "domesctic_emissions_construction (2026-2050)",
        "construction_emissions_tonnes_EU": "EU_construction_emissions_tonnes (2026-2050)",
        "construction_emissions_tonnes_non_EU" : "non_EU_construction_emissions_tonnes (2026-2050)"
        
        })
    )

# Dataframe with construction result totals

# Total imported emissions in the EU every year for every tech
df_total_construction_emissions_EU_logistic = (
    annual_construction_emissions_location_logistic
    .groupby(["tech", "year"], as_index=False)[["construction_emissions_tonnes_self","construction_emissions_tonnes_EU", "construction_emissions_tonnes_non_EU", "construction_emissions_tonnes_total"]]
    .sum()
    )

# Total imported emissions for each tech
df_total_construction_emissions_techs_logistic = (
    df_total_construction_emissions_EU_logistic
    .groupby(["tech"], as_index=False)[["construction_emissions_tonnes_self","construction_emissions_tonnes_EU","construction_emissions_tonnes_non_EU", "construction_emissions_tonnes_total"]]
    .sum()
    .rename(columns={
        "construction_emissions_tonnes_total": "total_construction_emissions_tonnes (2026-2050)",
        "construction_emissions_tonnes_self" : "domesctic_emissions_construction (2026-2050)",
        "construction_emissions_tonnes_EU": "EU_construction_emissions_tonnes (2026-2050)",
        "construction_emissions_tonnes_non_EU" : "non_EU_construction_emissions_tonnes (2026-2050)"
        
        })
    )
# Dataframe with construction result totals

# Total imported emissions in the EU every year for every tech
df_total_construction_emissions_EU_gompertz = (
    annual_construction_emissions_location_gompertz
    .groupby(["tech", "year"], as_index=False)[["construction_emissions_tonnes_self","construction_emissions_tonnes_EU", "construction_emissions_tonnes_non_EU", "construction_emissions_tonnes_total"]]
    .sum()
    )

# Total imported emissions for each tech
df_total_construction_emissions_techs_gompertz = (
    df_total_construction_emissions_EU_gompertz
    .groupby(["tech"], as_index=False)[["construction_emissions_tonnes_self","construction_emissions_tonnes_EU","construction_emissions_tonnes_non_EU", "construction_emissions_tonnes_total"]]
    .sum()
    .rename(columns={
        "construction_emissions_tonnes_total": "total_construction_emissions_tonnes (2026-2050)",
        "construction_emissions_tonnes_self" : "domesctic_emissions_construction (2026-2050)",
        "construction_emissions_tonnes_EU": "EU_construction_emissions_tonnes (2026-2050)",
        "construction_emissions_tonnes_non_EU" : "non_EU_construction_emissions_tonnes (2026-2050)"
        
        })
    )

In [112]:
annual_total_emissions_location_linear = (
annual_construction_emissions_location_linear
.merge(
    annual_imported_emissions_location_linear,
        on=["country", "tech", "year"],
        how="left"))


annual_total_emissions_location_linear["total_self"]= annual_total_emissions_location_linear["construction_emissions_tonnes_self"] + annual_total_emissions_location_linear["imported_emissions_tonnes_self"]
annual_total_emissions_location_linear["total_EU"]= annual_total_emissions_location_linear["construction_emissions_tonnes_EU"] + annual_total_emissions_location_linear["imported_emissions_tonnes_EU"]
annual_total_emissions_location_linear["total_non_EU"]= annual_total_emissions_location_linear["construction_emissions_tonnes_non_EU"] + annual_total_emissions_location_linear["imported_emissions_tonnes_non_EU"]
annual_total_emissions_location_linear["total"]= annual_total_emissions_location_linear["construction_emissions_tonnes_total"] + annual_total_emissions_location_linear["imported_emissions_tonnes_total"]


annual_total_emissions_location_linear = annual_total_emissions_location_linear[
    [
        "country",
        "tech",
        "year",
        "total_self",
        "total_EU",
        "total_non_EU",
        "total"
    ]
].copy()

annual_total_emissions_location_logistic = (
annual_construction_emissions_location_logistic
.merge(
    annual_imported_emissions_location_logistic,
        on=["country", "tech", "year"],
        how="left"))


annual_total_emissions_location_logistic["total_self"]= annual_total_emissions_location_logistic["construction_emissions_tonnes_self"] + annual_total_emissions_location_logistic["imported_emissions_tonnes_self"]
annual_total_emissions_location_logistic["total_EU"]= annual_total_emissions_location_logistic["construction_emissions_tonnes_EU"] + annual_total_emissions_location_logistic["imported_emissions_tonnes_EU"]
annual_total_emissions_location_logistic["total_non_EU"]= annual_total_emissions_location_logistic["construction_emissions_tonnes_non_EU"] + annual_total_emissions_location_logistic["imported_emissions_tonnes_non_EU"]
annual_total_emissions_location_logistic["total"]= annual_total_emissions_location_logistic["construction_emissions_tonnes_total"] + annual_total_emissions_location_logistic["imported_emissions_tonnes_total"]


annual_total_emissions_location_logistic = annual_total_emissions_location_logistic[
    [
        "country",
        "tech",
        "year",
        "total_self",
        "total_EU",
        "total_non_EU",
        "total"
    ]
].copy()

annual_total_emissions_location_gompertz = (
annual_construction_emissions_location_gompertz
.merge(
    annual_imported_emissions_location_gompertz,
        on=["country", "tech", "year"],
        how="left"))


annual_total_emissions_location_gompertz["total_self"]= annual_total_emissions_location_gompertz["construction_emissions_tonnes_self"] + annual_total_emissions_location_gompertz["imported_emissions_tonnes_self"]
annual_total_emissions_location_gompertz["total_EU"]= annual_total_emissions_location_gompertz["construction_emissions_tonnes_EU"] + annual_total_emissions_location_gompertz["imported_emissions_tonnes_EU"]
annual_total_emissions_location_gompertz["total_non_EU"]= annual_total_emissions_location_gompertz["construction_emissions_tonnes_non_EU"] + annual_total_emissions_location_gompertz["imported_emissions_tonnes_non_EU"]
annual_total_emissions_location_gompertz["total"]= annual_total_emissions_location_gompertz["construction_emissions_tonnes_total"] + annual_total_emissions_location_gompertz["imported_emissions_tonnes_total"]


annual_total_emissions_location_gompertz = annual_total_emissions_location_gompertz[
    [
        "country",
        "tech",
        "year",
        "total_self",
        "total_EU",
        "total_non_EU",
        "total"
    ]
].copy()

In [113]:
total_EU_Emissions_locations_linear = (
annual_total_emissions_location_linear
.groupby(["tech", "year"], as_index=False)[["total_self","total_EU","total_non_EU"]]
.sum()
    )

total_EU_Emissions_locations_logistic = (
annual_total_emissions_location_logistic
.groupby(["tech", "year"], as_index=False)[["total_self","total_EU","total_non_EU"]]
.sum()
    )

total_EU_Emissions_locations_gompertz = (
annual_total_emissions_location_gompertz
.groupby(["tech", "year"], as_index=False)[["total_self","total_EU","total_non_EU"]]
.sum()
    )

In [116]:
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

# EU consuming countries used in your model
EU_CONSUMING_COUNTRIES = set(EU_countries)

# EU origin countries used to define whether upstream emissions are inside or outside the EU.
# Add Malta here if it exists in the MRIO data and should not be counted as non-EU.
EU_ORIGIN_COUNTRIES = set(EU_countries) | {"MLT"}

OUTPUT_DIR = Path("outputs/upstream_origin_tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Growth model input dataframes
growth_model_investments = {
    "linear": df_annual_investment_linear,
    "logistic": df_annual_investment_logistics,
    "gompertz": df_annual_investment_gompertz,
}


# ------------------------------------------------------------
# Optional mapping tables for readable names
# ------------------------------------------------------------

# Sector names from GHG file, if available
if {"nace_r2_code", "nace_r2"}.issubset(df_GHG.columns):
    sector_name_map = (
        df_GHG[["nace_r2_code", "nace_r2"]]
        .dropna()
        .drop_duplicates(subset=["nace_r2_code"])
        .set_index("nace_r2_code")["nace_r2"]
        .to_dict()
    )
else:
    sector_name_map = {}

# Region names from regions file, if a readable column exists
possible_region_name_cols = [
    col for col in df_regions.columns
    if col.lower() in ["name", "country", "country_name", "region", "region_name"]
]

if "ISO_3" in df_regions.columns and len(possible_region_name_cols) > 0:
    region_name_col = possible_region_name_cols[0]
    region_name_map = (
        df_regions[["ISO_3", region_name_col]]
        .dropna()
        .drop_duplicates(subset=["ISO_3"])
        .set_index("ISO_3")[region_name_col]
        .to_dict()
    )
else:
    region_name_map = {}


# ------------------------------------------------------------
# Prepare Leontief-based non-EU upstream emission coefficients
# ------------------------------------------------------------

def prepare_non_eu_upstream_coefficients():
    """
    Creates a dataframe with Leontief-based upstream emission coefficients
    for non-EU origin countries only.

    Each row represents emissions caused in country_orig / sector_orig
    per euro of final output in country_dest / sector_dest.
    """

    coeff = df_upstream_emissions_per_euro.copy()

    coeff["upstream_emissions_per_euro"] = (
        coeff["upstream_emissions_per_euro"]
        .fillna(0)
    )

    # Keep only emissions that occur outside the EU
    coeff = coeff.loc[
        ~coeff["country_orig"].isin(EU_ORIGIN_COUNTRIES)
    ].copy()

    # Only keep the sectors relevant for this analysis:
    # C27 = solar and batteries
    # C28 = wind
    # F   = construction
    relevant_destination_sectors = set(res_code.values())

    coeff = coeff.loc[
        coeff["sector_dest"].isin(relevant_destination_sectors)
    ].copy()

    return coeff


df_non_eu_upstream_coeff = prepare_non_eu_upstream_coefficients()


# ------------------------------------------------------------
# Prepare annual investment dataframe
# ------------------------------------------------------------

def prepare_investment_df(df_investment):
    """
    Standardises the annual investment dataframe.

    Onshore and offshore wind are grouped into 'wind', because the MRIO
    technology-sector mapping uses 'wind' as one product group.
    """

    df = df_investment.copy()

    df["tech"] = df["tech"].replace({
        "onshore wind": "wind",
        "offshore wind": "wind"
    })

    df = df.loc[
        df["country"].isin(EU_CONSUMING_COUNTRIES)
    ].copy()

    df["investment"] = df["investment"].fillna(0)
    df["construction_cost"] = df["construction_cost"].fillna(0)

    return df


# ------------------------------------------------------------
# Calculate non-EU upstream emissions by origin region and sector
# ------------------------------------------------------------

def calculate_non_eu_origin_contributions(df_investment, model_name):
    """
    Calculates the origin of non-EU upstream emissions for one growth model.

    It includes:
    1. equipment/component investment emissions, allocated through import shares;
    2. construction emissions, allocated to sector F in the consuming EU country.

    Both parts use the Leontief-based upstream emissions per euro and only
    count emissions occurring outside the EU.
    """

    df_inv = prepare_investment_df(df_investment)

    # --------------------------------------------------------
    # 1. Equipment / imported technology component demand
    # --------------------------------------------------------

    equipment_demand = (
        df_inv
        .groupby(["country", "tech", "year"], as_index=False)["investment"]
        .sum()
    )

    equipment_demand["sector_dest"] = equipment_demand["tech"].map(res_code)

    # Add import shares:
    # country = EU consuming country
    # country_orig in df_import_shares = direct import partner / producer country
    equipment_demand = equipment_demand.merge(
        df_import_shares[
            ["country_dest", "country_orig", "tech", "import_share"]
        ],
        left_on=["country", "tech"],
        right_on=["country_dest", "tech"],
        how="left",
        validate="many_to_many"
    )

    # Diagnostics for missing import shares
    missing_import_shares = equipment_demand.loc[
        equipment_demand["import_share"].isna(),
        ["country", "tech"]
    ].drop_duplicates()

    if len(missing_import_shares) > 0:
        print(f"\nWarning: missing import shares in {model_name}:")
        print(missing_import_shares)

    equipment_demand["import_share"] = equipment_demand["import_share"].fillna(0)

    equipment_demand["weighted_equipment_demand_eur"] = (
        equipment_demand["investment"] *
        equipment_demand["import_share"]
    )

    equipment_demand = equipment_demand.rename(columns={
        "country_orig": "producer_country"
    })

    # Aggregate before merging with Leontief coefficients to avoid unnecessary rows
    equipment_demand_agg = (
        equipment_demand
        .groupby(["producer_country", "sector_dest"], as_index=False)
        ["weighted_equipment_demand_eur"]
        .sum()
        .rename(columns={
            "producer_country": "country_dest",
            "weighted_equipment_demand_eur": "final_demand_eur"
        })
    )

    equipment_demand_agg["demand_type"] = "equipment"

    # --------------------------------------------------------
    # 2. Construction demand
    # --------------------------------------------------------

    construction_demand_agg = (
        df_inv
        .groupby(["country"], as_index=False)["construction_cost"]
        .sum()
        .rename(columns={
            "country": "country_dest",
            "construction_cost": "final_demand_eur"
        })
    )

    construction_demand_agg["sector_dest"] = res_code["construction"]
    construction_demand_agg["demand_type"] = "construction"

    # --------------------------------------------------------
    # 3. Combine final demand and apply Leontief coefficients
    # --------------------------------------------------------

    total_demand = pd.concat(
        [
            equipment_demand_agg[
                ["country_dest", "sector_dest", "final_demand_eur", "demand_type"]
            ],
            construction_demand_agg[
                ["country_dest", "sector_dest", "final_demand_eur", "demand_type"]
            ]
        ],
        ignore_index=True
    )

    total_demand = total_demand.loc[
        total_demand["final_demand_eur"] > 0
    ].copy()

    contributions = total_demand.merge(
        df_non_eu_upstream_coeff[
            [
                "country_dest",
                "sector_dest",
                "country_orig",
                "sector_orig",
                "upstream_emissions_per_euro"
            ]
        ],
        on=["country_dest", "sector_dest"],
        how="left",
        validate="many_to_many"
    )

    contributions["upstream_emissions_per_euro"] = (
        contributions["upstream_emissions_per_euro"].fillna(0)
    )

    # Same unit logic as your earlier code:
    # final demand [EUR] * emissions per euro / 1000 = tonnes CO2-eq
    contributions["non_eu_upstream_emissions_tonnes"] = (
        contributions["final_demand_eur"] *
        contributions["upstream_emissions_per_euro"]
        / 1000
    )

    contributions = contributions.loc[
        contributions["non_eu_upstream_emissions_tonnes"] > 0
    ].copy()

    total_non_eu_emissions = (
        contributions["non_eu_upstream_emissions_tonnes"].sum()
    )

    # --------------------------------------------------------
    # 4. Region table
    # --------------------------------------------------------

    region_table = (
        contributions
        .groupby("country_orig", as_index=False)["non_eu_upstream_emissions_tonnes"]
        .sum()
        .rename(columns={
            "country_orig": "region",
            "non_eu_upstream_emissions_tonnes": "emissions_tonnes"
        })
    )

    region_table["emissions_MtCO2eq"] = (
        region_table["emissions_tonnes"] / 1e6
    )

    region_table["percentage"] = np.where(
        total_non_eu_emissions > 0,
        region_table["emissions_tonnes"] / total_non_eu_emissions * 100,
        0
    )

    region_table["region_name"] = (
        region_table["region"].map(region_name_map).fillna(region_table["region"])
    )

    region_table = region_table[
        ["region", "region_name", "emissions_MtCO2eq", "percentage"]
    ].sort_values("percentage", ascending=False)

    # --------------------------------------------------------
    # 5. Industry table
    # --------------------------------------------------------

    industry_table = (
        contributions
        .groupby("sector_orig", as_index=False)["non_eu_upstream_emissions_tonnes"]
        .sum()
        .rename(columns={
            "sector_orig": "industry",
            "non_eu_upstream_emissions_tonnes": "emissions_tonnes"
        })
    )

    industry_table["emissions_MtCO2eq"] = (
        industry_table["emissions_tonnes"] / 1e6
    )

    industry_table["percentage"] = np.where(
        total_non_eu_emissions > 0,
        industry_table["emissions_tonnes"] / total_non_eu_emissions * 100,
        0
    )

    industry_table["industry_name"] = (
        industry_table["industry"].map(sector_name_map).fillna(industry_table["industry"])
    )

    industry_table = industry_table[
        ["industry", "industry_name", "emissions_MtCO2eq", "percentage"]
    ].sort_values("percentage", ascending=False)

    # --------------------------------------------------------
    # 6. Save output
    # --------------------------------------------------------

    region_table.to_csv(
        OUTPUT_DIR / f"non_eu_upstream_region_share_{model_name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    industry_table.to_csv(
        OUTPUT_DIR / f"non_eu_upstream_industry_share_{model_name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    return region_table, industry_table, contributions


# ------------------------------------------------------------
# Run for all three growth models
# ------------------------------------------------------------

non_eu_region_tables = {}
non_eu_industry_tables = {}
non_eu_contribution_details = {}

for model_name, df_model in growth_model_investments.items():

    region_table, industry_table, contributions = calculate_non_eu_origin_contributions(
        df_model,
        model_name
    )

    non_eu_region_tables[model_name] = region_table
    non_eu_industry_tables[model_name] = industry_table
    non_eu_contribution_details[model_name] = contributions

    print(f"\n{model_name.upper()} - top 10 non-EU origin regions")
    display(region_table.head(10))

    print(f"\n{model_name.upper()} - top 10 non-EU origin industries")
    display(industry_table.head(10))


      country       tech
20050     FRA  batteries
20075     FRA      solar
20100     FRA       wind

LINEAR - top 10 non-EU origin regions


,region,region_name,emissions_MtCO2eq,percentage
13,ROW,ROW,45.365328,52.734014
5,CHN,CHN,16.340450,18.994628
14,RUS,RUS,5.581399,6.487985
8,IND,IND,2.584626,3.004447
16,TUR,TUR,2.367147,2.751642
6,GBR,GBR,2.261100,2.628371
18,ZAF,ZAF,1.979963,2.301569
17,USA,USA,1.660279,1.929957
12,NOR,NOR,1.586088,1.843717
15,SAU,SAU,1.181208,1.373072



LINEAR - top 10 non-EU origin industries


,industry,industry_name,emissions_MtCO2eq,percentage
3,B,Mining and quarrying,24.149462,28.072057
14,C24,Manufacture of basic metals,22.864500,26.578379
23,D35,"Electricity, gas, steam and air conditioning s...",16.425741,19.093772
10,C20,Manufacture of chemicals and chemical products,5.205424,6.050941
13,C23,Manufacture of other non-metallic mineral prod...,3.188958,3.706940
25,E37T39,"Sewerage, waste management, remediation activi...",2.762498,3.211211
31,H50,Water transport,1.985219,2.307678
18,C28,Manufacture of machinery and equipment n.e.c.,1.805057,2.098253
17,C27,Manufacture of electrical equipment,1.421694,1.652620
30,H49,Land transport and transport via pipelines,1.339730,1.557342



      country       tech
20050     FRA  batteries
20075     FRA      solar
20100     FRA       wind

LOGISTIC - top 10 non-EU origin regions


...

,industry,industry_name,emissions_MtCO2eq,percentage
3,B,Mining and quarrying,23.713733,28.075987
14,C24,Manufacture of basic metals,22.441051,26.569189
23,D35,"Electricity, gas, steam and air conditioning s...",16.124861,19.091106
10,C20,Manufacture of chemicals and chemical products,5.115122,6.056072
13,C23,Manufacture of other non-metallic mineral prod...,3.133219,3.709589
25,E37T39,"Sewerage, waste management, remediation activi...",2.710818,3.209485
31,H50,Water transport,1.948586,2.307037
18,C28,Manufacture of machinery and equipment n.e.c.,1.765459,2.090224
17,C27,Manufacture of electrical equipment,1.405094,1.663567
30,H49,Land transport and transport via pipelines,1.315312,1.557270



      country       tech
20050     FRA  batteries
20075     FRA      solar
20100     FRA       wind

GOMPERTZ - top 10 non-EU origin regions


,region,region_name,emissions_MtCO2eq,percentage
13,ROW,ROW,46.567028,52.733163
5,CHN,CHN,16.782623,19.004880
14,RUS,RUS,5.728017,6.486488
8,IND,IND,2.649686,3.000542
16,TUR,TUR,2.429008,2.750643
6,GBR,GBR,2.320683,2.627974
18,ZAF,ZAF,2.031658,2.300678
17,USA,USA,1.704438,1.930129
12,NOR,NOR,1.627024,1.842465
15,SAU,SAU,1.212657,1.373230



GOMPERTZ - top 10 non-EU origin industries


,industry,industry_name,emissions_MtCO2eq,percentage
3,B,Mining and quarrying,24.791288,28.074006
14,C24,Manufacture of basic metals,23.463820,26.570763
23,D35,"Electricity, gas, steam and air conditioning s...",16.859929,19.092423
10,C20,Manufacture of chemicals and chemical products,5.347175,6.055217
13,C23,Manufacture of other non-metallic mineral prod...,3.275581,3.709315
25,E37T39,"Sewerage, waste management, remediation activi...",2.834353,3.209662
31,H50,Water transport,2.037344,2.307117
18,C28,Manufacture of machinery and equipment n.e.c.,1.847386,2.092006
17,C27,Manufacture of electrical equipment,1.467201,1.661480
30,H49,Land transport and transport via pipelines,1.375137,1.557224


In [117]:
region_table.to_csv(
    OUTPUT_DIR / f"non_eu_upstream_region_share_{model_name}.csv",
    index=False,
    encoding="utf-8-sig"
)

industry_table.to_csv(
    OUTPUT_DIR / f"non_eu_upstream_industry_share_{model_name}.csv",
    index=False,
    encoding="utf-8-sig"
)